In [1]:
# !pip install pinecone sentence-transformers
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

In [2]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Loading the API keys
gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [4]:
#Connect to the pinecone
pinecone = Pinecone(api_key=pinecone_api_key)

In [5]:
INDEX_NAME = "ragtest"

pinecone.create_index(
    name=INDEX_NAME,
    dimension=384,  #Depends on your embedding Model
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

ConflictError: [409 ALREADY_EXISTS] Resource  already exists

In [6]:
# ---------- upsert: simple documents ----------
# Step 3: Chunking and organizing documents - Here I'm giving you already organized docs.
# This is where most hard work is done.
docs = [
    {"id": "doc1", "text": "Pandas is a Python library for data analysis."},
    {"id": "doc2", "text": "Pinecone is a vector database for semantic search."},
    {"id": "doc3", "text": "Spark enables distributed data processing."},
]

docs

[{'id': 'doc1', 'text': 'Pandas is a Python library for data analysis.'},
 {'id': 'doc2', 'text': 'Pinecone is a vector database for semantic search.'},
 {'id': 'doc3', 'text': 'Spark enables distributed data processing.'}]

In [7]:
# embed in a small batch (better for throughput than per-doc calls)
# Step 4 - Creating Embeddings
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")  # 384-dim
embeddings = model.encode([d["text"] for d in docs])  # shape (n, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
# Step 6 - Organize vectors with ids/metadata
vectors = [
    {
        "id": d["id"],
        "values": emb.tolist(),
        "metadata": {"text": d["text"]},
    }
    for d, emb in zip(docs, embeddings)
]

In [9]:
vectors = []
for d, emb in zip(docs,embeddings):
    print(d["id"])
    print(emb)
    print(d["text"])
    vectors.append()

doc1
[-4.88402337e-01 -7.72713363e-01 -1.79367423e-01  2.75081277e-01
  1.05562091e-01 -2.61944443e-01  5.11257887e-01 -1.22027636e-01
  1.51373312e-01  4.51361090e-01 -1.83275506e-01 -3.89374226e-01
  2.13705018e-01  3.72340590e-01 -4.15670484e-01 -2.12128926e-02
  3.46354634e-01 -5.76614030e-02 -2.83594728e-02 -5.01845777e-01
 -4.71999884e-01 -5.06300665e-02 -2.25893274e-01  3.88020515e-01
 -2.10162655e-01 -5.84237397e-01 -2.57924408e-01 -4.91476357e-02
  6.05109744e-02  4.01333012e-02  1.42069474e-01 -2.38506794e-01
 -6.56093955e-02  3.72613788e-01 -4.34653550e-01  2.55043656e-01
 -5.17540090e-02 -3.30621362e-01 -1.25788167e-01  1.37097389e-01
 -3.84487152e-01  3.62321138e-01  6.59914553e-01  2.60308325e-01
  1.28738403e-01 -6.78920299e-02 -2.76502877e-01 -2.70921350e-01
 -3.35516334e-01  1.44570291e-01 -2.01469004e-01  1.21743359e-01
 -4.19496387e-01 -8.15767527e-01  1.56849816e-01  2.36290265e-02
  5.59956431e-01 -4.45928834e-02 -1.78022325e-01 -4.78786558e-01
  1.64695233e-01  9.

TypeError: list.append() takes exactly one argument (0 given)

In [18]:
INDEX_NAME = "ragtest"
index = pinecone.Index(INDEX_NAME)

In [19]:
index

Index(host='https://ragtest-503dlyx.svc.aped-4627-b74a.pinecone.io')

In [20]:
index.upsert(vectors)

TypeError: Index.upsert() takes 1 positional argument but 2 were given

In [21]:
# Input query
query = "what tool help us in searching"
qvec = model.encode([query])[0].tolist()

In [22]:
qvec

[-0.5059787034988403,
 0.038043878972530365,
 0.11087412387132645,
 -0.1493634581565857,
 0.31026023626327515,
 0.0024359002709388733,
 0.43681830167770386,
 -0.12117551267147064,
 0.0825711339712143,
 -0.05501726642251015,
 0.5904010534286499,
 -0.3692326545715332,
 0.060185547918081284,
 -0.010740403085947037,
 -0.040332648903131485,
 -0.4952974021434784,
 -0.18015927076339722,
 -0.012215843424201012,
 -0.08699028939008713,
 -0.21831713616847992,
 0.26601409912109375,
 0.4404066205024719,
 -0.21311423182487488,
 -0.40803229808807373,
 -0.0595218725502491,
 0.045135170221328735,
 -0.4132850766181946,
 -0.41921737790107727,
 0.35486626625061035,
 -0.2048977166414261,
 0.3243890106678009,
 -0.2060835063457489,
 -0.24244453012943268,
 0.20012912154197693,
 0.6376379728317261,
 0.12253475934267044,
 -0.21842753887176514,
 -0.12223827093839645,
 -0.02185608632862568,
 -0.35854560136795044,
 -0.07941675931215286,
 -0.1513010561466217,
 -0.07289157807826996,
 0.6106327772140503,
 0.008092490

In [40]:
# get the closest value from vector DB
res = index.query(vector=qvec, top_k=3, include_metadata=True)
res

QueryResponse(matches=[{'id': 'doc2',
 'metadata': {'text': 'Pinecone is a vector database for semantic search.'},
 'score': 0.441973776,
 'values': []}, {'id': 'doc1',
 'metadata': {'text': 'Pandas is a Python library for data analysis.'},
 'score': 0.27415064,
 'values': []}, {'id': 'doc3',
 'metadata': {'text': 'Spark enables distributed data processing.'},
 'score': 0.18384032,
 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sat, 06 Dec 2025 06:56:10 GMT', 'content-type': 'application/json', 'content-length': '407', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '5', 'x-pinecone-request-id': '4293482691835456713', 'x-envoy-upstream-service-time': '5', 'grpc-status': '0', 'server': 'envoy'}})

# Let's be back by 12:50